In [4]:
import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
import helpers
import heuristic
from heuristic import Path, VGeneralizedSolver
from solver import solve_pdpcr

import pickle
import copy
import datetime
from docplex.mp.model import Model
import logging
import heapq
import networkx as nx
import numpy as np
import time

logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

In [6]:

total_robots = 10
random_data = False

data_type = 'csv'
# data_name = 'generated18.csv'
# data_name = 'data3.csv'
data_name = 'lim/100/lrc203-5-uniform.csv'
# data_name = 'lim/200/LC1_2_3-5-uniform.csv'
total_tasks = 6 # random data

raw = get_data(data_name, data_type)
logging.info(f'Solving data {data_name} with {total_robots} robots and {len(raw.tasks)} tasks')
processed_h = heuristic.process_greedy_1(raw)

depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)

In [9]:
greedy_v2 = heuristic.GreedyInsert(processed_h, c_travel, c_operate)
greddy_v2_F, greedy_v2_C_max = greedy_v2.solve(total_robots)
print(f'greedy insert makespan: {greedy_v2_C_max}')

greedy insert makespan: 920.7543117467869


In [ ]:
#Li larger data

first_names = ['LC1']
sec_names = [2,4,6,8,10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [10]:
#Li smaller data
small_first_names =['lc','lr','lrc']
small_sec_names = [100+i for i in range(1,10)] + [200 + i for i in range(1,10)]
data_paths = [f'lim/100/{i}{j}-5-uniform.csv' for i in small_first_names for j in small_sec_names]

In [11]:
num_robots = [10]
file_type = 'csv'
is_depot_task = False
#new greedy search
for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            tik1 = time.time()
            greedy_v2 = heuristic.GreedyInsert(processed_h, c_travel, c_operate)
            greedy_v2_F, greedy_v2_C_max = greedy_v2.solve(num_robot)
            tok1 = time.time()
            greedy_time = tok1 - tik1
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
                pickle.dump(greedy_v2_F, open(f'solve_results/greedy2/{data_name}.p', 'wb'))
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(greedy_v2_C_max)
            solve_log.append(np.round(greedy_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            with open('solve_log_greedy2.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)
            raise(e)

25-Aug-24 03:04:02:INFO:Solving data lim/100/lc101-5-uniform.csv with 10 robots and 53 tasks



 ______________Error in lim/100/lc101-5-uniform.csv with 10 robots______________

name 'total_robots' is not defined


NameError: name 'total_robots' is not defined

In [3]:
first_names = ['LC1']
sec_names = [2,4,6]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [11]:
num_robots = [10]
file_type = 'csv'
is_depot_task = False

for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            cur_best_makespan = np.infty
            for i in range(30):
                solve_viet = VGeneralizedSolver(processed_h, c_travel, c_operate)
                cur_sol = solve_viet.solve(num_robot,'random',cfg.TABU_INIT_CONSIDER_PROCESS_TIME,task_range=[1,4])
                makespan = cur_sol.get_makespan()
    
                if makespan < cur_best_makespan:
                    init_sol  = cur_sol
                    cur_best_makespan = makespan
            
            #save init_sol as file knowing that init_sol is an instance of Solution class
            data_name = data_path.split('/')[2].split('.')[0]
            pickle.dump(init_sol, open(f'solve_results/init_data/{data_name}.p', 'wb'))

        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)
            raise(e)

17-Sep-24 15:54:49:INFO:Solving data lim/200/LC1_2_1-5-uniform.csv with 10 robots and 106 tasks


Viet generalized sol is 941.44
Viet generalized sol is 1005.68
Viet generalized sol is 942.21
Viet generalized sol is 938.78
Viet generalized sol is 1007.8
Viet generalized sol is 938.81
Viet generalized sol is 1051.2
Viet generalized sol is 943.41
Viet generalized sol is 1062.85
Viet generalized sol is 997.34
Viet generalized sol is 986.55
Viet generalized sol is 955.79
Viet generalized sol is 1102.7
Viet generalized sol is 1101.21
Viet generalized sol is 940.33
Viet generalized sol is 1216.85
Viet generalized sol is 973.27
Viet generalized sol is 1055.57
Viet generalized sol is 948.25
Viet generalized sol is 977.05
Viet generalized sol is 990.93
Viet generalized sol is 956.17
Viet generalized sol is 1010.45
Viet generalized sol is 940.85
Viet generalized sol is 937.25
Viet generalized sol is 1108.15
Viet generalized sol is 1027.11
Viet generalized sol is 1019.11
Viet generalized sol is 999.89


17-Sep-24 15:54:52:INFO:Solving data lim/200/LC1_2_2-5-uniform.csv with 10 robots and 105 tasks


Viet generalized sol is 943.72
Viet generalized sol is 1261.14
Viet generalized sol is 1357.64
Viet generalized sol is 1362.48
Viet generalized sol is 1377.18
Viet generalized sol is 1122.17
Viet generalized sol is 1331.5
Viet generalized sol is 1366.3
Viet generalized sol is 1288.14
Viet generalized sol is 1313.69
Viet generalized sol is 1447.65
Viet generalized sol is 1391.28
Viet generalized sol is 1157.18
Viet generalized sol is 1239.04
Viet generalized sol is 1211.95
Viet generalized sol is 1310.89
Viet generalized sol is 1184.14
Viet generalized sol is 1405.08
Viet generalized sol is 1397.57
Viet generalized sol is 1121.39
Viet generalized sol is 1231.49
Viet generalized sol is 1243.62
Viet generalized sol is 1300.71
Viet generalized sol is 1467.43
Viet generalized sol is 1265.28
Viet generalized sol is 1410.12
Viet generalized sol is 1365.7
Viet generalized sol is 1256.44
Viet generalized sol is 1346.7


17-Sep-24 15:54:56:INFO:Solving data lim/200/LC1_2_3-5-uniform.csv with 10 robots and 103 tasks


Viet generalized sol is 1218.11
Viet generalized sol is 1235.71
Viet generalized sol is 1530.95
Viet generalized sol is 1732.44
Viet generalized sol is 1620.74
Viet generalized sol is 1697.19
Viet generalized sol is 1701.88
Viet generalized sol is 1667.68
Viet generalized sol is 1669.37
Viet generalized sol is 1660.52
Viet generalized sol is 1474.53
Viet generalized sol is 1534.94
Viet generalized sol is 1675.97
Viet generalized sol is 1680.75
Viet generalized sol is 1662.55
Viet generalized sol is 1603.52
Viet generalized sol is 1551.12
Viet generalized sol is 1617.59
Viet generalized sol is 1569.51
Viet generalized sol is 1477.49
Viet generalized sol is 1465.66
Viet generalized sol is 1581.98
Viet generalized sol is 1690.6
Viet generalized sol is 1510.56
Viet generalized sol is 1541.55
Viet generalized sol is 1831.9
Viet generalized sol is 1687.08
Viet generalized sol is 1739.87
Viet generalized sol is 1718.26
Viet generalized sol is 1710.95
Viet generalized sol is 1607.9


... (50 outputs left out)

17-Sep-24 16:00:27:INFO:Solving data lim/600/LC1_6_9-5-uniform.csv with 10 robots and 316 tasks


Viet generalized sol is 4071.11
Viet generalized sol is 3930.05
Viet generalized sol is 4375.4
Viet generalized sol is 4284.54
Viet generalized sol is 4361.08
Viet generalized sol is 4224.69
Viet generalized sol is 4136.64
Viet generalized sol is 4392.15
Viet generalized sol is 4439.56
Viet generalized sol is 4289.65
Viet generalized sol is 4120.44
Viet generalized sol is 4347.25
Viet generalized sol is 4129.95
Viet generalized sol is 4315.47
Viet generalized sol is 3851.0
Viet generalized sol is 4262.63
Viet generalized sol is 4197.02
Viet generalized sol is 4240.3
Viet generalized sol is 4526.94
Viet generalized sol is 4397.15
Viet generalized sol is 3762.95
Viet generalized sol is 4419.63
Viet generalized sol is 4076.7
Viet generalized sol is 4079.39
Viet generalized sol is 3933.86
Viet generalized sol is 4120.59
Viet generalized sol is 3993.92
Viet generalized sol is 4037.95
Viet generalized sol is 4556.6
Viet generalized sol is 4062.26


17-Sep-24 16:00:51:INFO:Solving data lim/600/LC1_6_10-5-uniform.csv with 10 robots and 319 tasks


Viet generalized sol is 4054.17
Viet generalized sol is 4369.16
Viet generalized sol is 4255.47
Viet generalized sol is 4630.03
Viet generalized sol is 4264.65
Viet generalized sol is 4368.18
Viet generalized sol is 4440.78
Viet generalized sol is 4344.69
Viet generalized sol is 4589.01
Viet generalized sol is 4771.47
Viet generalized sol is 4346.65
Viet generalized sol is 4301.33
Viet generalized sol is 4723.49
Viet generalized sol is 4417.39
Viet generalized sol is 4475.94
Viet generalized sol is 4750.68
Viet generalized sol is 4678.97
Viet generalized sol is 4375.42
Viet generalized sol is 4769.0
Viet generalized sol is 4156.82
Viet generalized sol is 4674.09
Viet generalized sol is 4379.16
Viet generalized sol is 4504.63
Viet generalized sol is 4772.95
Viet generalized sol is 4324.76
Viet generalized sol is 4465.09
Viet generalized sol is 4288.88
Viet generalized sol is 4230.27
Viet generalized sol is 4500.38
Viet generalized sol is 4297.61
Viet generalized sol is 4509.53
